# Network Anomaly Detection on UNSW-NB15: Random Forest vs XGBoost


**Task:** binary classification, 0 = normal traffic, 1 = attack.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

## 2. Load data

In [ ]:
train = pd.read_csv("UNSW_NB15_training-set.csv", encoding="utf-8-sig")
test  = pd.read_csv("UNSW_NB15_testing-set.csv",  encoding="utf-8-sig")
print(train.shape, test.shape)
print(train["label"].value_counts())
train.head()

(175341, 45) (82332, 45)
label
1    119341
0     56000
Name: count, dtype: int64


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,1,0,0,0,1,1,0,Normal,0
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,1,2,0,0,0,1,6,0,Normal,0
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,1,3,0,0,0,2,6,0,Normal,0
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,...,1,3,1,1,0,2,1,0,Normal,0
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,...,1,40,0,0,0,2,39,0,Normal,0


## 3. Preprocess (as in the paper)

### Drop id (not useful) and attack_cat (attack name, would leak the answer)

In [ ]:
y_train = train["label"]
y_test  = test["label"]
X_train = train.drop(columns=["id", "attack_cat", "label"])
X_test  = test.drop(columns=["id", "attack_cat", "label"])
print("Features:", X_train_s.shape[1])

### Label-encode proto, service, state

In [ ]:
# Label encoding (fit on both sets so unseen categories don't cause errors)
for col in ["proto", "service", "state"]:
    le = LabelEncoder().fit(pd.concat([X_train[col], X_test[col]]))
    X_train[col] = le.transform(X_train[col])
    X_test[col]  = le.transform(X_test[col])

### Min-Max scale all features to [0, 1]

In [ ]:
# Scaling (fit on train only)
scaler = MinMaxScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns)

## 4. Train the two models
Settings follow the paper: Random Forest with 100 trees, XGBoost with default settings and `logloss`.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb = XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1)

rf.fit(X_train_s, y_train)
xgb.fit(X_train_s, y_train)
print("Training done")

Training done


## 5. Evaluate on the test set

In [ ]:
def evaluate(model, name):
    pred = model.predict(X_test_s)
    prob = model.predict_proba(X_test_s)[:, 1]
    return {"Model": name,
            "Accuracy":  accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred),
            "Recall":    recall_score(y_test, pred),
            "F1-Score":  f1_score(y_test, pred),
            "AUC":       roc_auc_score(y_test, prob)}

results = pd.DataFrame([evaluate(rf, "Random Forest"), evaluate(xgb, "XGBoost")]).set_index("Model")
results.round(4)

,Accuracy,Precision,Recall,F1-Score,AUC
Model,,,,,
Random Forest,0.8726,0.8196,0.9856,0.8950,0.9783
XGBoost,0.8738,0.8238,0.9806,0.8954,0.9827


## 6. Charts

In [ ]:
results.T.plot(kind="bar", figsize=(8, 4))
plt.title("Random Forest vs XGBoost")
plt.ylabel("Score")
plt.ylim(0.5, 1.0)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

NameError: name 'results' is not defined